### Notebook 3 — Jointures et agrégations

In [49]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, year, datediff, col, round, concat_ws
from pyspark.sql.functions import sum, desc, countDistinct, date_trunc, asc, avg, dense_rank
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.sql.functions import trim, initcap, upper, col
from collections import Counter
from pyspark.sql.window import Window

##### SparkSession

In [50]:
spark = SparkSession.builder.appName("TradeCorp ETL Notebook3").getOrCreate()
spark

##### chargement de parquet files pour la transformation

In [51]:
#Lecture simple d’un Parquet

DATA_PATH = "/home/jovyan/data/bronze"   # adapte selon ton volume Docker

# Lecture des fichiers Parquet
df_categories = spark.read.parquet(f"{DATA_PATH}/categories/")
df_customers = spark.read.parquet(f"{DATA_PATH}/customers_clean/")
df_employees = spark.read.parquet(f"{DATA_PATH}/employees_clean/")
df_suppliers = spark.read.parquet(f"{DATA_PATH}/supplies_silver/")
df_shippers = spark.read.parquet(f"{DATA_PATH}/shippers_silver/")
df_order_details = spark.read.parquet(f"{DATA_PATH}/order_details_final/")
df_orders = spark.read.parquet(f"{DATA_PATH}/orders_1997/")
df_products = spark.read.parquet(f"{DATA_PATH}/products_filtered/")

# Dictionnaire des DataFrames
dfs = {
    "customers_clean": df_customers,
    "employees_clean": df_employees,
    "suppliers_silver": df_suppliers,
    "shippers_silver": df_shippers,
    "order_details_final": df_order_details,
    "orders_1997": df_orders,
    "categories": df_categories,
    "products_filtered": df_products,    
}

In [52]:
# EDA vérifier nulls

def count_nulls(df, name):
    results = []
    
    for c in df.columns:
        null_count = df.filter(col(c).isNull()).count()
        results.append((name, c, null_count))
    
    return spark.createDataFrame(results, ["dataframe", "column", "null_count"])

# Dictionnaire des DataFrames
dfs = {
    "customers_clean": df_customers,
    "employees_clean": df_employees,
    "suppliers_silver": df_suppliers,
    "shippers_silver": df_shippers,
    "order_details_final": df_order_details,
    "orders_1997": df_orders,
    "categories": df_categories,
    "products_filtered": df_products,    
}
summary_nulls = None

for name, df in dfs.items():
    df_nulls = count_nulls(df, name)
    summary_nulls = df_nulls if summary_nulls is None else summary_nulls.unionByName(df_nulls)
summary_nulls.count()

70

##### Écriture en Parquet (Silver layer)

##### A21 — Jointure orders + customers

*Joindre df_orders et df_customers sur customer_id. Garder uniquement : order_id, company_name, country,
order_date, freight.*

In [53]:
# Joindre df_orders et df_customers sur customer_id. Garder uniquement : order_id, company_name, country, order_date, freight
df_orders_customers = (
    df_orders.alias("o")
    .join(df_customers.alias("c"), on="customer_id", how="inner")
    .select(
        col("o.order_id"),
        col("c.company_name"),
        col("c.country"),
        col("o.order_date"),
        col("o.freight")
    )
)

#Vérification
df_orders_customers.show(10)
df_orders_customers.printSchema()

+--------+--------------------+---------+----------+-------+
|order_id|        company_name|  country|order_date|freight|
+--------+--------------------+---------+----------+-------+
|   10400|  Eastern Connection|       UK|1997-01-01|  83.93|
|   10401|Rattlesnake Canyo...|      USA|1997-01-01|  12.51|
|   10402|        Ernst Handel|  AUSTRIA|1997-01-02|  67.88|
|   10403|        Ernst Handel|  AUSTRIA|1997-01-03|  73.79|
|   10404|Magazzini Aliment...|    ITALY|1997-01-03| 155.97|
|   10405|    LINO-Delicateses|VENEZUELA|1997-01-06|  34.82|
|   10406|       Queen Cozinha|   BRAZIL|1997-01-07| 108.04|
|   10407|  Ottilies Käseladen|  GERMANY|1997-01-07|  91.48|
|   10408|   Folies gourmandes|   FRANCE|1997-01-08|  11.26|
|   10409|Océano Atlántico ...|ARGENTINA|1997-01-09|  29.83|
+--------+--------------------+---------+----------+-------+
only showing top 10 rows

root
 |-- order_id: integer (nullable = true)
 |-- company_name: string (nullable = true)
 |-- country: string (nullable

##### A22 — Jointure order_details + products

*Joindre df_order_details et df_products sur product_id. Ajouter les colonnes product_name, category_id,
unit_price depuis products.*

In [54]:
df_order_details_products = (
    df_order_details.alias("od")
    .join(df_products.alias("p"), on="product_id", how="inner")
    .select(
        "od.order_id",
        "od.product_id",
        "od.prix_unitaire",
        "od.quantite",
        "od.discount",
        "od.sous_total",
        "p.product_name",
        "p.category_id",
        "p.unit_price"
    )
)
#Vérification
#df_order_details_products.show(10)
df_order_details_products.show(3, truncate=False)

+--------+----------+-------------+--------+--------+----------+----------------------+-----------+----------+
|order_id|product_id|prix_unitaire|quantite|discount|sous_total|product_name          |category_id|unit_price|
+--------+----------+-------------+--------+--------+----------+----------------------+-----------+----------+
|10248   |11        |14.0         |12      |0.0     |168.0     |Queso Cabrales        |4          |21.0      |
|10248   |72        |34.8         |5       |0.0     |174.0     |Mozzarella di Giovanni|4          |34.8      |
|10249   |14        |18.6         |9       |0.0     |167.4     |Tofu                  |7          |23.25     |
+--------+----------+-------------+--------+--------+----------+----------------------+-----------+----------+
only showing top 3 rows



##### Q23 ointure products + categories

*Joindre df_products et df_categories sur category_id pour enrichir chaque produit avec category_name et
description.*

In [55]:
# Joindre df_products et df_categories sur category_id pour enrichir chaque produit avec category_name et description.

df_products_categories = (
    df_products.alias("p")
    .join(df_categories.alias("c"), on="category_id", how="left")
    .select(
        "p.product_id",
        "p.product_name",
        "p.supplier_id",
        "p.category_id",
        "c.category_name",
        "c.description",
        "p.quantity_per_unit",
        "p.unit_price",
        "p.units_in_stock",
        "p.units_on_order",
        "p.reorder_level",
        "p.discontinued"
    )
)
#Vérification

#df_products_categories.show(10)
df_products_categories.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- supplier_id: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity_per_unit: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- units_in_stock: integer (nullable = true)
 |-- units_on_order: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- discontinued: integer (nullable = true)



##### Q24 DataFrame enrichi complet


*A. Réaliser une première jointure complète (order_details, orders, customers, products enrichi avec categories, employees, shippers) sans renommer aucune colonne. Lister ensuite les colonnes qui apparaissenten double grâce à Counter.*


In [56]:
# products enriched

# 1. Chargement et préparation des tables avec renommage préfixé pour éviter les doublons
df_customers_renamed = df_customers \
    .withColumnRenamed("country", "customer_country") \
    .withColumnRenamed("city", "customer_city") \
    .withColumnRenamed("phone", "customer_phone")

df_employees_renamed = df_employees \
    .withColumnRenamed("country", "employee_country") \
    .withColumnRenamed("city", "employee_city") \
    .withColumnRenamed("unit_price", "employee_unit_price") # si besoin

df_shippers_renamed = df_shippers.withColumnRenamed("company_name", "shipper_name")

df_products_renamed = df_products.withColumnRenamed("unit_price", "product_unit_price")
df_categories_renamed = df_categories.withColumnRenamed("description", "category_description")

df_products_full = df_products_renamed.join(df_categories_renamed, on="category_id", how="inner")

# 2. Re-jointure complète avec les tables renommées
df_full_clean = df_order_details \
    .join(df_orders, on="order_id", how="inner") \
    .join(df_customers_renamed, on="customer_id", how="inner") \
    .join(df_employees_renamed, on="employee_id", how="inner") \
    .join(df_shippers_renamed, on="shipper_id", how="inner") \
    .join(df_products_full, on="product_id", how="inner")

*B. Pour chaque colonne identifiée en Q24a, la renommer dans sa table d'origine avant de refaire la jointure, en la préfixant selon la table (customer_country, employee_country, shipper_name...). Reconstruire ensuite df_orders_enriched avec ces tables renommées, puis vérifier qu'il ne reste plus aucun doublon.*

In [57]:
# Vérification

df_products_enriched_raw = df_products.join(df_categories, on="category_id", how="inner")

df_full_raw = df_order_details \
    .join(df_orders, on="order_id", how="inner") \
    .join(df_customers, on="customer_id", how="inner") \
    .join(df_employees, on="employee_id", how="inner") \
    .join(df_shippers, on="shipper_id", how="inner") \
    .join(df_products_enriched_raw, on="product_id", how="inner")

# 3. Analyse des occurrences de chaque nom de colonne avec Counter
col_counts = Counter(df_full_raw.columns)

print("Colonnes en double (Q24a)")
for col_name, count in col_counts.items():
    if count > 1:
        print(f"Colonne '{col_name}' : présente {count} fois")

Colonnes en double (Q24a)
Colonne 'company_name' : présente 2 fois
Colonne 'city' : présente 2 fois
Colonne 'country' : présente 2 fois
Colonne 'phone' : présente 2 fois


In [58]:
# Vérification des doublons restants
col_counts = Counter(df_full_clean.columns)
remaining_duplicates = [c for c, count in col_counts.items() if count > 1]

print("Doublons restants :", remaining_duplicates)
print("Nombre total de colonnes :", len(df_full_clean.columns))

Doublons restants : []
Nombre total de colonnes : 52


In [59]:
PATH = "/home/jovyan/data/silver"

df_full_clean.write.mode("overwrite").option("header", "true").csv(f"{PATH}/orders_enriched_csv")

##### Q25 — CA par client

*Calculer le chiffre d'affaires total par client (company_name) depuis df_orders_enriched. Trier par CA
décroissant. Afficher le top 10*

In [60]:
#for gold data_parquet

PATH = "/home/jovyan/data/silver"

df_full = spark.read.csv(f"{PATH}/orders_enriched_csv", header=True, inferSchema=True)
df_full.show(3, truncate=False)

+----------+----------+-----------+-----------+--------+-------------+--------+--------+----------+----------+-------------+------------+-------+----------------------------+------------------+---------+-----------+----------------+------------+----------+----------------------------+----------------+------------------+------------------+-------------+--------+-----------+----------------+--------------+-----------+----------+---------+--------------------+----------+-------------+----------------+---------------+----------------+--------------+-----------+--------------------------------+-----------+-----------------+------------------+--------------+--------------+-------------+------------+--------+-------------+----------------------------------------------------------+-------+
|product_id|shipper_id|employee_id|customer_id|order_id|prix_unitaire|quantite|discount|sous_total|order_date|required_date|shipped_date|freight|ship_name                   |ship_address      |ship_city|ship

In [61]:
df_full.withColumn("ca", col("quantite").cast("double") \
                         * col("product_unit_price").cast("double") \
                            * (1 - col("discount").cast("double"))) \
                                .groupBy("company_name") \
                                    .agg(round(sum("ca"),2).alias("CA")) \
                                        .orderBy(desc("CA")) \
                                            .show(10, truncate=False)

+----------------------------+--------+
|company_name                |CA      |
+----------------------------+--------+
|QUICK-Stop                  |52872.11|
|Ernst Handel                |42269.0 |
|Save-a-lot Markets          |41072.24|
|Mère Paillarde              |25440.4 |
|Rattlesnake Canyon Grocery  |21045.02|
|Simons bistro               |19029.51|
|Hungry Owl All-Night Grocers|14764.03|
|Folk och fä HB              |13325.23|
|HILARION-Abastos            |12953.66|
|Folies gourmandes           |12173.7 |
+----------------------------+--------+
only showing top 10 rows



##### Q26 — CA par catégorie
*Calculer le CA total par catégorie de produits. Afficher le nombre de produits distincts vendus par catégorie.*

In [62]:
# Q26 — CA total et produits distincts vendus par catégorie
df_full.withColumn("ca", col("quantite").cast("double")
                         * col("prix_unitaire").cast("double")
                         * (1 - col("discount").cast("double"))) \
                            .groupBy("category_name") \
                                .agg(round(sum("ca"),2).alias("ca_total"),
                                     countDistinct("product_id").alias("nb_produits_distincts")) \
                                        .orderBy(desc("ca_total")) \
                                            .show(truncate=False)

+--------------+---------+---------------------+
|category_name |ca_total |nb_produits_distincts|
+--------------+---------+---------------------+
|Dairy Products|108086.89|9                    |
|Beverages     |90368.63 |9                    |
|Confections   |82657.75 |13                   |
|Seafood       |66959.22 |12                   |
|Condiments    |54994.97 |11                   |
|Grains/Cereals|51463.63 |6                    |
|Produce       |40992.09 |4                    |
|Meat/Poultry  |11017.16 |2                    |
+--------------+---------+---------------------+



##### Q27 — CA par mois
*Calculer le CA mensuel. Utiliser date_trunc ou month() et year() pour extraire le mois et l'année*

In [63]:
# Q27 — CA mensuel par ordre chronologique
df_full.withColumn("ca", col("quantite").cast("double")
                         *col("prix_unitaire").cast("double")
                         *(1 - col("discount").cast("double"))) \
                            .withColumn("mois", date_trunc("month", col("order_date"))) \
                                .groupBy("mois") \
                                    .agg(round(sum("ca"),2).alias("ca_mensuel")) \
                                        .orderBy(asc("mois")) \
                                            .show(10, truncate=False)

+-------------------+----------+
|mois               |ca_mensuel|
+-------------------+----------+
|1997-01-01 00:00:00|51487.5   |
|1997-02-01 00:00:00|31549.04  |
|1997-03-01 00:00:00|33226.32  |
|1997-04-01 00:00:00|41510.6   |
|1997-05-01 00:00:00|48895.27  |
|1997-06-01 00:00:00|29875.45  |
|1997-07-01 00:00:00|45162.86  |
|1997-08-01 00:00:00|38039.93  |
|1997-09-01 00:00:00|43335.4   |
|1997-10-01 00:00:00|48574.49  |
+-------------------+----------+
only showing top 10 rows



##### Q28 — Performance par employé
*Calculer pour chaque employé (full_name) : le nombre de commandes traitées, le CA total généré et le délai
moyen de livraison en jours.*

In [64]:
from pyspark.sql.functions import col, when, count

df_full.groupBy("full_name") \
    .agg(countDistinct("order_id").alias("nb_commandes")) \
        .show()

+----------------+------------+
|       full_name|nb_commandes|
+----------------+------------+
|  Anne Dodsworth|          18|
|   Nancy Davolio|          54|
|   Andrew Fuller|          40|
| Steven Buchanan|          18|
| Janet Leverling|          71|
|     Robert King|          33|
|  Laura Callahan|          53|
|Margaret Peacock|          75|
|  Michael Suyama|          33|
+----------------+------------+



In [65]:
df_full.withColumn("ca", col("quantite").cast("double")
                         * col("prix_unitaire").cast("double")
                         * (1 - col("discount").cast("double"))) \
                            .withColumn("delai_livraison", datediff(col("shipped_date"), col("order_date"))) \
                                .groupBy("full_name") \
                                    .agg(count("order_id").alias("nb_commandes"), \
                                         round(sum("ca"),2).alias("ca_total"), \
                                            round(avg("delai_livraison"),2).alias("delai_moyen_jours")) \
                                                .orderBy(desc("ca_total")) \
                                                    .show(truncate=False)

+----------------+------------+---------+-----------------+
|full_name       |nb_commandes|ca_total |delai_moyen_jours|
+----------------+------------+---------+-----------------+
|Margaret Peacock|182         |104193.75|8.3              |
|Janet Leverling |165         |97081.26 |8.92             |
|Nancy Davolio   |137         |81898.88 |7.84             |
|Andrew Fuller   |84          |54907.03 |10.19            |
|Robert King     |74          |49562.79 |9.81             |
|Laura Callahan  |103         |47077.94 |7.95             |
|Michael Suyama  |71          |34037.06 |7.94             |
|Anne Dodsworth  |38          |20595.99 |10.03            |
|Steven Buchanan |39          |17185.64 |6.46             |
+----------------+------------+---------+-----------------+



##### Q29 — Window functions — Rang
*Classer les produits par CA généré avec dense_rank(). Utiliser une Window partitionnée par category_name.*

In [66]:
df_produits_ca = df_full.withColumn("ca", col("quantite").cast("double")
                                          * col("prix_unitaire").cast("double")
                                          * (1 - col("discount").cast("double"))) \
                                            .groupBy("category_name", "product_name") \
                                                .agg(round(sum("ca"),2).alias("ca"))

In [67]:
window_spec = Window.partitionBy("category_name").orderBy(desc("ca"))

df_produits_ca.withColumn("rang", dense_rank().over(window_spec)) \
    .orderBy("category_name", "rang") \
    .show(10, truncate=False)


+-------------+--------------------------------+--------+----+
|category_name|product_name                    |ca      |rang|
+-------------+--------------------------------+--------+----+
|Beverages    |Côte de Blaye                   |49198.09|1   |
|Beverages    |Ipoh Coffee                     |11069.9 |2   |
|Beverages    |Lakkalikööri                    |7379.1  |3   |
|Beverages    |Outback Lager                   |5468.4  |4   |
|Beverages    |Steeleye Stout                  |5274.9  |5   |
|Beverages    |Rhönbräu Klosterbier            |4485.55 |6   |
|Beverages    |Chartreuse verte                |4475.7  |7   |
|Beverages    |Sasquatch Ale                   |2107.0  |8   |
|Beverages    |Laughing Lumberjack Lager       |910.0   |9   |
|Condiments   |Louisiana Fiery Hot Pepper Sauce|9373.19 |1   |
+-------------+--------------------------------+--------+----+
only showing top 10 rows



##### Q30 — Window functions — Cumul
*Calculer le CA cumulé par mois (ordre chronologique) avec sum() sur une Window orderBy date.*

In [68]:
df_ca_mois = df_full.withColumn("ca", col("quantite").cast("double")
                                      * col("prix_unitaire").cast("double")
                                      * (1 - col("discount").cast("double"))) \
                                        .withColumn("mois", date_trunc("month", col("order_date"))) \
                                            .groupBy("mois") \
                                                .agg(round(sum("ca"),2).alias("ca_mensuel"))

window_cumul = Window.orderBy(asc("mois"))

df_ca_mois.withColumn("ca_cumule", round(sum("ca_mensuel").over(window_cumul),2)) \
    .orderBy(asc("mois")) \
    .show(20, truncate=False)

+-------------------+----------+---------+
|mois               |ca_mensuel|ca_cumule|
+-------------------+----------+---------+
|1997-01-01 00:00:00|51487.5   |51487.5  |
|1997-02-01 00:00:00|31549.04  |83036.54 |
|1997-03-01 00:00:00|33226.32  |116262.86|
|1997-04-01 00:00:00|41510.6   |157773.46|
|1997-05-01 00:00:00|48895.27  |206668.73|
|1997-06-01 00:00:00|29875.45  |236544.18|
|1997-07-01 00:00:00|45162.86  |281707.04|
|1997-08-01 00:00:00|38039.93  |319746.97|
|1997-09-01 00:00:00|43335.4   |363082.37|
|1997-10-01 00:00:00|48574.49  |411656.86|
|1997-11-01 00:00:00|39898.78  |451555.64|
|1997-12-01 00:00:00|54984.69  |506540.33|
+-------------------+----------+---------+



##### Q31 — Tri et limite
- *Afficher les 5 produits les plus vendus en quantité (toutes commandes confondues).*
- *Afficher les 3 pays clients (customer_country) qui génèrent le plus de chiffre d'affaires.*

In [69]:
# 1. Top 5 des produits les plus vendus en quantité
df_full.groupBy("product_name") \
    .agg(sum("quantite").alias("total_quantite")) \
    .orderBy(desc("total_quantite")) \
    .show(5, truncate=False)

# 2. Top 3 des pays clients générant le plus de chiffre d'affaires
df_full.withColumn("ca", col("quantite").cast("double") * col("prix_unitaire").cast("double") * (1 - col("discount").cast("double"))) \
    .groupBy("customer_country") \
    .agg(sum("ca").alias("ca_total")) \
    .orderBy(desc("ca_total")) \
    .show(5, truncate=False)

+----------------------+--------------+
|product_name          |total_quantite|
+----------------------+--------------+
|Gnocchi di nonna Alice|971           |
|Raclette Courdavault  |752           |
|Camembert Pierrot     |665           |
|Rhönbräu Klosterbier  |630           |
|Sir Rodney's Scones   |610           |
+----------------------+--------------+
only showing top 5 rows

+----------------+------------------+
|customer_country|ca_total          |
+----------------+------------------+
|GERMANY         |100641.26250000001|
|USA             |90731.68250000001 |
|AUSTRIA         |46559.4875        |
|FRANCE          |40041.0875        |
|BRAZIL          |35940.04250000001 |
+----------------+------------------+
only showing top 5 rows



##### Q32 — Écriture en Parquet
*Écrire df_orders_enriched en format Parquet dans /home/jovyan/data/output/orders_enriched.parquet. Utiliser le
mode overwrite.*

In [70]:
PATH = "/home/jovyan/data/gold"
df_full.write.mode("overwrite").parquet(f"{PATH}/orders_enriched.parquet")

print("Écriture au format Parquet effectuée avec succès !")

Écriture au format Parquet effectuée avec succès !


In [71]:
### Pour comparer la taille de csv et parquet, les données sont stocké en csv aussi
PATH = "/home/jovyan/data/gold"

df_full.write.mode("overwrite").option("header", "true").csv(f"{PATH}/orders_enriched.csv")

### Finale Notebook 3